In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# PIPELINE JURNAL

In [ ]:
#@title Import Library
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow

In [ ]:
#@title Fungsi Load & Validasi Citra
def load_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Gambar tidak ditemukan / bukan gambar")
    return img

In [ ]:
#@title Preprocessing (BGR → HSV)
def convert_to_hsv(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return hsv

In [ ]:
#@title Segmentasi Warna (Masking)
def masking_hsv(hsv):
    mask_green = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_yellow = cv2.inRange(hsv, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_red = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
               cv2.inRange(hsv, np.array([160, 40, 40]), np.array([180, 255, 255]))

    return mask_green, mask_yellow, mask_red

In [ ]:
#@title Ekstraksi Fitur (Hitung Piksel)
def hitung_piksel(mask_green, mask_yellow, mask_red):
    count_g = np.sum(mask_green == 255)
    count_y = np.sum(mask_yellow == 255)
    count_r = np.sum(mask_red == 255)
    total = count_g + count_y + count_r

    return count_g, count_y, count_r, total

In [ ]:
#@title Hitung Persentase
def hitung_persentase(count_g, count_y, count_r, total):
    per_g = count_g / total if total > 0 else 0
    per_y = count_y / total if total > 0 else 0
    per_r = count_r / total if total > 0 else 0

    return per_g, per_y, per_r

In [ ]:
#@title Klasifikasi
def klasifikasi(per_g, per_y, per_r):
    if per_g > 0.5:
        return "Belum Matang"
    elif per_y > 0.6:
        return "Setengah Matang"
    else:
        return "Matang"

In [ ]:
#@title Pipeline Satu Gambar
def proses_satu_gambar(image_path):
    img = load_image(image_path)
    if img is None:
        return None, None

    hsv = convert_to_hsv(img)
    mask_g, mask_y, mask_r = masking_hsv(hsv)
    count_g, count_y, count_r, total = hitung_piksel(mask_g, mask_y, mask_r)
    per_g, per_y, per_r = hitung_persentase(count_g, count_y, count_r, total)
    hasil = klasifikasi(per_g, per_y, per_r)

    return hasil, img

In [ ]:
#@title Proses Seluruh Dataset
def proses_dataset(base_path):
    kategori_folder = ['Fresh', 'Rotten']
    statistik = {'total': 0, 'benar': 0}

    print(f"{'File':<30} | {'Aktual':<10} | {'Prediksi':<15} | {'Status'}")
    print("-" * 70)

    for kat in kategori_folder:
        folder_path = os.path.join(base_path, kat)
        if not os.path.exists(folder_path):
            continue

        for file_name in os.listdir(folder_path):
            full_path = os.path.join(folder_path, file_name)

            if os.path.isfile(full_path):
                hasil = proses_satu_gambar(full_path)

                if hasil is not None:
                    prediksi, _ = hasil

                    aktual = "Matang" if kat == "Fresh" else "Busuk/Belum Matang"
                    is_correct = "Lolos" if prediksi == "Matang" and kat == "Fresh" else "Gagal"

                    if is_correct == "Lolos":
                        statistik['benar'] += 1

                    statistik['total'] += 1

                    print(f"{file_name[:30]:<30} | {kat:<10} | {prediksi:<15} | {is_correct}")

    akurasi = (statistik['benar'] / statistik['total']) * 100 if statistik['total'] > 0 else 0

    print("-" * 70)
    print(f"Total Data : {statistik['total']}")
    print(f"Benar      : {statistik['benar']}")
    print(f"Akurasi    : {akurasi:.2f}%")

In [ ]:
dataset_root = '/content/drive/MyDrive/Apple/'
proses_dataset(dataset_root)

In [ ]:
#@title pipeline full
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow

def identifikasi_tunggal(image_path):
    """Fungsi inti untuk memproses satu gambar (Metode HSV Masking)"""
    img = cv2.imread(image_path)
    # Jika cv2.imread gagal membaca file (bukan gambar), ia akan mengembalikan None
    if img is None: return None, None

    # 2. Preprocessing & Transformasi
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 3. Segmentasi (Masking)
    mask_green = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_yellow = cv2.inRange(hsv, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_red = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
               cv2.inRange(hsv, np.array([160, 40, 40]), np.array([180, 255, 255]))

    # 4. Ekstraksi Fitur (Hitung Piksel)
    count_g = np.sum(mask_green == 255)
    count_y = np.sum(mask_yellow == 255)
    count_r = np.sum(mask_red == 255)
    total_pixel = count_g + count_y + count_r

    # 5. Hitung Persentase
    per_g = count_g / total_pixel if total_pixel > 0 else 0
    per_y = count_y / total_pixel if total_pixel > 0 else 0
    per_r = count_r / total_pixel if total_pixel > 0 else 0

    # 6. Klasifikasi
    if per_g > 0.5:
        return "Belum Matang", img
    elif per_y > 0.6:
        return "Setengah Matang" , img
    else:
        return "Matang", img

# --- BAGIAN BARU: PEMROSESAN BANYAK DATASET ---

def proses_seluruh_dataset(base_path):
    # Struktur folder diharapkan: /Apple/Fresh/ dan /Apple/Rotten/
    kategori_folder = ['Fresh', 'Rotten']
    statistik = {'total': 0, 'benar': 0}
    total_files_in_folders = 0
    processed_files_count = 0

    print(f"{'File':<30} | {'Aktual':<10} | {'Prediksi':<15} | {'Status'}")
    print("-" * 70)

    for kat in kategori_folder:
        folder_path = os.path.join(base_path, kat)
        if not os.path.exists(folder_path): continue

        files_in_current_folder = os.listdir(folder_path)
        total_files_in_folders += len(files_in_current_folder)

        for file_name in files_in_current_folder:
            full_path = os.path.join(folder_path, file_name)

            # PERUBAHAN DI SINI:
            # Mengecek apakah path adalah file (bukan folder) tanpa filter ekstensi
            if os.path.isfile(full_path):
                # Jalankan deteksi
                hasil = identifikasi_tunggal(full_path)

                # Cek jika file tersebut berhasil dibaca sebagai gambar
                if hasil is not None:
                    prediksi, _ = hasil
                    processed_files_count += 1

                    # Logika Validasi untuk Akurasi
                    aktual = "Matang" if kat == "Fresh" else "Busuk/Belum Matang"
                    is_correct = "Lolos" if prediksi == "Matang" and kat == "Fresh" else "Gagal"

                    if is_correct == "Lolos": statistik['benar'] += 1
                    statistik['total'] += 1

                    print(f"{file_name[:30]:<30} | {kat:<10} | {prediksi:<15} | {is_correct}")

    # Output Statistik Akhir untuk Laporan
    akurasi = (statistik['benar'] / statistik['total']) * 100 if statistik['total'] > 0 else 0
    print("-" * 70)
    print(f"Total file yang Ditemukan: {total_files_in_folders}")
    print(f"Total file Gambar yang Diproses: {processed_files_count}")
    print(f"Total Data Diuji : {statistik['total']}")
    print(f"Total Benar      : {statistik['benar']}")
    print(f"Akurasi Sistem   : {akurasi:.2f}%")

# Jalankan Pipeline
dataset_root = '/content/drive/MyDrive/Apple/'
proses_seluruh_dataset(dataset_root)

# PIPELINE MODIFIKASI JURNAL

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
#@title PREPROCESSING
def tahap_1_preprocessing(img_path):
    img = cv2.imread(img_path)
    if img is None: return None, None

    # Standarisasi ukuran agar penghitungan piksel konsisten
    img_resized = cv2.resize(img, (400, 400))
    hsv_img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    return img_resized, hsv_img

In [ ]:
#@title Ekstraksi Fitur (Segmentasi & Masking)
def tahap_2_feature_extraction(hsv_img):
    # Masking warna berdasarkan range HSV
    mask_g = cv2.inRange(hsv_img, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_y = cv2.inRange(hsv_img, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_r = cv2.inRange(hsv_img, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
             cv2.inRange(hsv_img, np.array([160, 40, 40]), np.array([180, 255, 255]))
    # Modifikasi: Masking Busuk (Brown)
    mask_b = cv2.inRange(hsv_img, np.array([0, 0, 0]), np.array([30, 255, 100]))

    # Ekstraksi angka dari citra (Piksel Count)
    count_g = np.sum(mask_g == 255)
    count_y = np.sum(mask_y == 255)
    count_r = np.sum(mask_r == 255)
    count_b = np.sum(mask_b == 255)
    total = count_g + count_y + count_r + count_b + 1

    fitur = {
        'per_g': count_g / total,
        'per_y': count_y / total,
        'per_r': count_r / total,
        'per_b': count_b / total
    }
    return fitur, mask_g, mask_y, mask_r, mask_b

In [ ]:
#@title Split Data
def tahap_3_split_data(base_path):
    all_paths = []
    all_labels = []

    for kat in ['Fresh', 'Rotten']:
        folder = os.path.join(base_path, kat)
        if not os.path.exists(folder): continue
        for f in os.listdir(folder):
            full_path = os.path.join(folder, f)
            if os.path.isfile(full_path):
                all_paths.append(full_path)
                all_labels.append(kat)

    # Stratify menjamin rasio kelas tetap seimbang di data training & testing
    train_p, test_p, train_l, test_l = train_test_split(
        all_paths, all_labels, test_size=0.20, stratify=all_labels, random_state=42
    )

    print(f"\n--- Data Split Statistics ---")
    print(f"Total Data: {len(all_paths)}")
    print(f"Data Training (80%): {len(train_p)}")
    print(f"Data Testing  (20%): {len(test_p)}")
    print(f"-----------------------------")

    return test_p, test_l # Mengembalikan data testing untuk diuji

In [ ]:
#@title Klasifikasi
def tahap_4_classification(fitur):
    # Logika Prioritas: Keamanan Pangan (SDG 2)
    if fitur['per_b'] > 0.12:
        prediksi = "Rotten"
    elif fitur['per_g'] > 0.5:
        prediksi = "Belum Matang"
    elif fitur['per_y'] > 0.6:
        prediksi = "Setengah Matang"
    else:
        prediksi = "Matang"

    # Penentuan Kelayakan
    if prediksi == "Rotten":
        kelayakan = "Tidak Layak Konsumsi"
    elif prediksi == "Matang":
        kelayakan = "Layak Konsumsi"
    else:
        kelayakan = "Perlu Pemeriksaan"

    return prediksi, kelayakan

In [ ]:
#@title Output (Visualisasi & Statistik)
def tampilkan_output(img, masks, info):
    prediksi, kelayakan, file_name, status = info
    mask_g, mask_y, mask_r, mask_b = masks

    fig, axes = plt.subplots(1, 5, figsize=(18,4))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original\n{file_name[:15]}")
    axes[1].imshow(mask_g, cmap='gray'); axes[1].set_title("Mask Hijau")
    axes[2].imshow(mask_y, cmap='gray'); axes[2].set_title("Mask Kuning")
    axes[3].imshow(mask_r, cmap='gray'); axes[3].set_title("Mask Merah")
    axes[4].imshow(mask_b, cmap='gray'); axes[4].set_title("Mask Busuk")

    for ax in axes: ax.axis('off')
    plt.suptitle(f"Hasil: {prediksi} | {kelayakan} | Status: {status}", fontsize=12)
    plt.show()

In [ ]:
#@title Pipeline Utama
def run_pipeline(dataset_path):
    # Jalankan Split Data
    test_paths, test_labels = tahap_3_split_data(dataset_path)
    statistik = {'total': 0, 'benar': 0}

    print(f"{'File':<30} | {'Aktual':<10} | {'Prediksi':<15} | {'Status'}")
    print("-" * 75)

    for path, aktual in zip(test_paths, test_labels):
        # Jalankan Preprocessing
        img, hsv = tahap_1_preprocessing(path)
        if hsv is None: continue

        # Jalankan Feature Extraction
        fitur, mg, my, mr, mb = tahap_2_feature_extraction(hsv)

        # Jalankan Klasifikasi
        prediksi, kelayakan = tahap_4_classification(fitur)

        # Validasi Status
        if aktual == 'Fresh':
            status = "Lolos" if prediksi == "Matang" else "Gagal"
        else:
            status = "Lolos" if prediksi == "Rotten" else "Gagal"

        if status == "Lolos": statistik['benar'] += 1
        statistik['total'] += 1

        # Tampilkan Output
        print(f"{os.path.basename(path)[:30]:<30} | {aktual:<10} | {prediksi:<15} | {status}")
        info_output = (prediksi, kelayakan, os.path.basename(path), status)
        tampilkan_output(img, (mg, my, mr, mb), info_output)

    # Final Statistik
    akurasi = (statistik['benar'] / statistik['total']) * 100
    print(f"\n[HASIL AKHIR] Akurasi pada Data Testing (20%): {akurasi:.2f}%")

# Eksekusi
dataset_root = '/content/drive/MyDrive/Apple'
run_pipeline(dataset_root)

In [ ]:
#@title pipeline full
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# TAHAP 1: PREPROCESSING
def tahap_1_preprocessing(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None, None

    img_resized = cv2.resize(img, (400, 400))
    hsv_img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    return img_resized, hsv_img


# TAHAP 2: FEATURE EXTRACTION + MASK
def tahap_2_feature_extraction(hsv_img):
    mask_g = cv2.inRange(hsv_img, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_y = cv2.inRange(hsv_img, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_r = cv2.inRange(hsv_img, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
             cv2.inRange(hsv_img, np.array([160, 40, 40]), np.array([180, 255, 255]))
    mask_b = cv2.inRange(hsv_img, np.array([0, 0, 0]), np.array([30, 255, 100]))

    count_g = np.sum(mask_g == 255)
    count_y = np.sum(mask_y == 255)
    count_r = np.sum(mask_r == 255)
    count_b = np.sum(mask_b == 255)

    total = count_g + count_y + count_r + count_b + 1

    fitur = {
        'per_g': count_g / total,
        'per_y': count_y / total,
        'per_r': count_r / total,
        'per_b': count_b / total
    }

    return fitur, mask_g, mask_y, mask_r, mask_b


# TAHAP 3: CLASSIFICATION
def tahap_3_classification(fitur):
    if fitur['per_b'] > 0.05:
        return "Rotten"
    elif fitur['per_g'] > 0.5:
        return "Belum Matang"
    elif fitur['per_y'] > 0.6:
        return "Setengah Matang"
    else:
        return "Matang"


# TAHAP 4: KELAYAKAN
def kelayakan_konsumsi(label):
    if label == "Rotten":
        return "Tidak Layak Konsumsi"
    elif label == "Matang":
        return "Layak Konsumsi"
    else:
        return "Perlu Pemeriksaan"


# VISUALISASI MASK
def tampilkan_visualisasi(img, mask_g, mask_y, mask_r, mask_b, prediksi):
    fig, axes = plt.subplots(1, 5, figsize=(18,4))

    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")

    axes[1].imshow(mask_g, cmap='gray')
    axes[1].set_title("Mask Hijau")

    axes[2].imshow(mask_y, cmap='gray')
    axes[2].set_title("Mask Kuning")

    axes[3].imshow(mask_r, cmap='gray')
    axes[3].set_title("Mask Merah")

    axes[4].imshow(mask_b, cmap='gray')
    axes[4].set_title("Mask Busuk")

    for ax in axes:
        ax.axis('off')

    plt.suptitle(f"Hasil Deteksi: {prediksi}", fontsize=14)
    plt.show()


# PIPELINE UTAMA
def proses_dataset_modifikasi(base_path, tampilkan_gambar=True):
    kategori_folder = ['Fresh', 'Rotten']
    statistik = {'total': 0, 'benar': 0}

    print(f"{'File':<30} | {'Aktual':<10} | {'Prediksi':<15} | {'Kelayakan':<25} | {'Status'}")
    print("-" * 90)

    for kat in kategori_folder:
        folder_path = os.path.join(base_path, kat)
        if not os.path.exists(folder_path):
            continue

        for file_name in os.listdir(folder_path):
            full_path = os.path.join(folder_path, file_name)

            if os.path.isdir(full_path):
                continue

            img, hsv = tahap_1_preprocessing(full_path)
            if hsv is None:
                continue

            fitur, mask_g, mask_y, mask_r, mask_b = tahap_2_feature_extraction(hsv)
            prediksi = tahap_3_classification(fitur)
            kelayakan = kelayakan_konsumsi(prediksi)

            # Validasi
            if kat == 'Fresh':
                status = "Lolos" if prediksi == "Matang" else "Gagal"
            else:
                status = "Lolos" if prediksi == "Rotten" else "Gagal"

            if status == "Lolos":
                statistik['benar'] += 1

            statistik['total'] += 1

            print(f"{file_name[:30]:<30} | {kat:<10} | {prediksi:<15} | {kelayakan:<25} | {status}")

            # tampilkan visualisasi (opsional)
            if tampilkan_gambar:
                tampilkan_visualisasi(img, mask_g, mask_y, mask_r, mask_b, prediksi)

    akurasi = (statistik['benar'] / statistik['total']) * 100 if statistik['total'] > 0 else 0

    print("-" * 90)
    print(f"Akurasi: {akurasi:.2f}%")


# JALANKAN
dataset_root = '/content/drive/MyDrive/Apple'
proses_dataset_modifikasi(dataset_root, tampilkan_gambar=True)

# Perbandingan pipeline jurnal dan pipeline modifikasi

In [ ]:
#@title PERBANDINGAN PIPELINE (Jurnal vs Modifikasi)
import cv2
import numpy as np
import os

# ===============================
# PIPELINE JURNAL (ORIGINAL)
# ===============================
def pipeline_jurnal(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    mask_g = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_y = cv2.inRange(hsv, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_r = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
             cv2.inRange(hsv, np.array([160, 40, 40]), np.array([180, 255, 255]))

    count_g = np.sum(mask_g == 255)
    count_y = np.sum(mask_y == 255)
    count_r = np.sum(mask_r == 255)
    total = count_g + count_y + count_r

    per_g = count_g / total if total > 0 else 0
    per_y = count_y / total if total > 0 else 0

    if per_g > 0.5:
        return "Belum Matang"
    elif per_y > 0.6:
        return "Setengah Matang"
    else:
        return "Matang"


# ===============================
# PIPELINE MODIFIKASI
# ===============================
def pipeline_modifikasi(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None

    img = cv2.resize(img, (400, 400))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    mask_g = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([85, 255, 255]))
    mask_y = cv2.inRange(hsv, np.array([20, 40, 40]), np.array([34, 255, 255]))
    mask_r = cv2.inRange(hsv, np.array([0, 40, 40]), np.array([10, 255, 255])) + \
             cv2.inRange(hsv, np.array([160, 40, 40]), np.array([180, 255, 255]))

    # tambahan deteksi busuk
    mask_b = cv2.inRange(hsv, np.array([0, 0, 0]), np.array([30, 255, 100]))

    count_g = np.sum(mask_g == 255)
    count_y = np.sum(mask_y == 255)
    count_r = np.sum(mask_r == 255)
    count_b = np.sum(mask_b == 255)

    total = count_g + count_y + count_r + count_b + 1

    per_g = count_g / total
    per_y = count_y / total
    per_b = count_b / total

    # prioritas busuk
    if per_b > 0.05:
        return "Rotten"
    elif per_g > 0.5:
        return "Belum Matang"
    elif per_y > 0.6:
        return "Setengah Matang"
    else:
        return "Matang"


# ===============================
# PROSES DATASET + PERBANDINGAN
# ===============================
def bandingkan_pipeline(base_path):
    kategori_folder = ['Fresh', 'Rotten']

    statistik = {
        'jurnal_benar': 0,
        'modif_benar': 0,
        'total': 0
    }

    print(f"{'File':<25} | {'Aktual':<10} | {'Jurnal':<15} | {'Modifikasi':<15}")
    print("-"*75)

    for kat in kategori_folder:
        folder_path = os.path.join(base_path, kat)
        if not os.path.exists(folder_path):
            continue

        for file_name in os.listdir(folder_path):
            full_path = os.path.join(folder_path, file_name)

            if not os.path.isfile(full_path):
                continue

            hasil_jurnal = pipeline_jurnal(full_path)
            hasil_modif  = pipeline_modifikasi(full_path)

            if hasil_jurnal is None or hasil_modif is None:
                continue

            # label ground truth sederhana
            if kat == "Fresh":
                aktual = "Matang"
            else:
                aktual = "Rotten"

            # cek akurasi
            if hasil_jurnal == aktual:
                statistik['jurnal_benar'] += 1

            if hasil_modif == aktual:
                statistik['modif_benar'] += 1

            statistik['total'] += 1

            print(f"{file_name[:25]:<25} | {aktual:<10} | {hasil_jurnal:<15} | {hasil_modif:<15}")

    # hasil akhir
    acc_jurnal = (statistik['jurnal_benar'] / statistik['total']) * 100 if statistik['total'] else 0
    acc_modif  = (statistik['modif_benar'] / statistik['total']) * 100 if statistik['total'] else 0

    print("\n" + "="*50)
    print(f"Akurasi Jurnal     : {acc_jurnal:.2f}%")
    print(f"Akurasi Modifikasi : {acc_modif:.2f}%")
    print("="*50)


# JALANKAN
dataset_root = '/content/drive/MyDrive/Apple'
bandingkan_pipeline(dataset_root)